# Performance and Precision Benchmarking

**What's in this notebook?** This notebook benchmarks JAXVacua operations across precision settings and demonstrates the adaptive precision system.

**In this notebook, you will learn:**
- How to detect and switch between precision modes via `jvc.set_precision`
- Timing comparisons for key operations at different precisions
- Numerical accuracy loss when using `float32`
- JIT compilation overhead and scaling with $h^{2,1}$

**Backend support:**

| Backend | float64 | float32 | Complex | Status |
|---------|:-------:|:-------:|:-------:|--------|
| CPU     | ✓ | ✓ | ✓ | Full support |
| CUDA    | ✓ | ✓ | ✓ | Full support |
| Metal (Apple GPU) | ✗ | ✓ | ✗ | Falls back to CPU |

(*Created:* March 2026)

## Imports and environment

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import jax
import jax.numpy as jnp
from jax import vmap
import time

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({"figure.dpi": 200, "font.size": 10, "axes.labelsize": 11,
                      "figure.figsize": (7, 3), "legend.frameon": False,
                      "font.family": "serif",
                      "xtick.direction": "in", "ytick.direction": "in"})

import jaxvacua as jvc

print(f"JAX: {jax.__version__}, Backend: {jax.default_backend()}")
print(f"Precision: {jvc.precision}, FLOAT: {jvc.FLOAT}, COMPLEX: {jvc.COMPLEX}")

In [ ]:
jvc.set_precision("float32")

## The `set_precision` API

```python
jvc.set_precision("float32")   # fast, reduced precision
jvc.set_precision("float64")   # full precision (default)
```

In [ ]:
print(f"Default: precision={jvc.precision}, jnp.array(1.0).dtype={jnp.array(1.0).dtype}")

jvc.set_precision("float32")
print(f"float32: precision={jvc.precision}, jnp.array(1.0).dtype={jnp.array(1.0).dtype}")

jvc.set_precision("float64")
print(f"float64: precision={jvc.precision}, jnp.array(1.0).dtype={jnp.array(1.0).dtype}")

## Benchmark helper

In [ ]:
def benchmark(fn, *args, n_warmup=3, n_runs=20, label=""):
    for _ in range(n_warmup):
        r = fn(*args)
        if hasattr(r, 'block_until_ready'): r.block_until_ready()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        r = fn(*args)
        if hasattr(r, 'block_until_ready'): r.block_until_ready()
        times.append((time.perf_counter() - t0) * 1000)
    m, s = np.mean(times), np.std(times)
    if label: print(f"  {label:40s} {m:8.3f} ± {s:.3f} ms")
    return m, s

## Benchmark: float64

In [ ]:
jvc.set_precision("float64")
model_64 = jvc.FluxEFT(h12=2, model_ID=1, maximum_degree=2, limit="LCS", model_type="KS")

z64 = jnp.array([0.1+2j, 0.2+3j])
zc64 = jnp.conj(z64); tau64 = jnp.array(0.1+5j); tauc64 = jnp.conj(tau64)
fl64 = jnp.array(np.random.default_rng(42).integers(-5, 6, 12).astype(float))

print(f"Precision: {jvc.precision}, z.dtype={z64.dtype}")
ops_64 = {}
ops_64["superpotential"] = benchmark(model_64.superpotential, z64, tau64, fl64, label="superpotential")
ops_64["DW"] = benchmark(model_64.DW, z64, zc64, tau64, tauc64, fl64, label="DW")
ops_64["kahler_metric"] = benchmark(model_64.kahler_metric, z64, zc64, tau64, tauc64, label="kahler_metric")
ops_64["hessian"] = benchmark(lambda: model_64.hessian(z64, zc64, tau64, tauc64, fl64), label="hessian (autodiff)")
ops_64["mass_matrix"] = benchmark(lambda: model_64.mass_matrix(z64, zc64, tau64, tauc64, fl64), label="mass_matrix")

In [ ]:
print(f"Precision: {jvc.precision}, z.dtype={z64.dtype}")
ops_64 = {}
ops_64["superpotential"] = benchmark(model_64.superpotential, z64, tau64, fl64, label="superpotential")
ops_64["DW"] = benchmark(model_64.DW, z64, zc64, tau64, tauc64, fl64, label="DW")
ops_64["kahler_metric"] = benchmark(model_64.kahler_metric, z64, zc64, tau64, tauc64, label="kahler_metric")
ops_64["hessian"] = benchmark(lambda: model_64.hessian(z64, zc64, tau64, tauc64, fl64), label="hessian (autodiff)")
ops_64["mass_matrix"] = benchmark(lambda: model_64.mass_matrix(z64, zc64, tau64, tauc64, fl64), label="mass_matrix")

## Benchmark: float32

In [ ]:
jvc.set_precision("float32")
model_32 = jvc.FluxEFT(h12=2, model_ID=1, maximum_degree=2, limit="LCS", model_type="KS")

z32 = jnp.array([0.1+2j, 0.2+3j])
zc32 = jnp.conj(z32); tau32 = jnp.array(0.1+5j); tauc32 = jnp.conj(tau32)
fl32 = jnp.array(np.random.default_rng(42).integers(-5, 6, 12).astype(float))

print(f"Precision: {jvc.precision}, z.dtype={z32.dtype}")
ops_32 = {}
ops_32["superpotential"] = benchmark(model_32.superpotential, z32, tau32, fl32, label="superpotential")
ops_32["DW"] = benchmark(model_32.DW, z32, zc32, tau32, tauc32, fl32, label="DW")
ops_32["kahler_metric"] = benchmark(model_32.kahler_metric, z32, zc32, tau32, tauc32, label="kahler_metric")
ops_32["hessian"] = benchmark(lambda: model_32.hessian(z32, zc32, tau32, tauc32, fl32), label="hessian (autodiff)")
ops_32["mass_matrix"] = benchmark(lambda: model_32.mass_matrix(z32, zc32, tau32, tauc32, fl32), label="mass_matrix")

#jvc.set_precision("float64")

In [ ]:
print(f"Precision: {jvc.precision}, z.dtype={z32.dtype}")
ops_32 = {}
ops_32["superpotential"] = benchmark(model_32.superpotential, z32, tau32, fl32, label="superpotential")
ops_32["DW"] = benchmark(model_32.DW, z32, zc32, tau32, tauc32, fl32, label="DW")
ops_32["kahler_metric"] = benchmark(model_32.kahler_metric, z32, zc32, tau32, tauc32, label="kahler_metric")
ops_32["hessian"] = benchmark(lambda: model_32.hessian(z32, zc32, tau32, tauc32, fl32), label="hessian (autodiff)")
ops_32["mass_matrix"] = benchmark(lambda: model_32.mass_matrix(z32, zc32, tau32, tauc32, fl32), label="mass_matrix")

### Speed comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))

op_names = list(ops_64.keys())
x = np.arange(len(op_names))
w = 0.35

m64 = [ops_64[op][0] for op in op_names]
m32 = [ops_32[op][0] for op in op_names]
s64 = [ops_64[op][1] for op in op_names]
s32 = [ops_32[op][1] for op in op_names]

ax.bar(x - w/2, m64, w, yerr=s64, label="float64", color="#2563eb", alpha=0.7, edgecolor="white", capsize=3)
ax.bar(x + w/2, m32, w, yerr=s32, label="float32", color="#f97316", alpha=0.7, edgecolor="white", capsize=3)

for i in range(len(op_names)):
    sp = m64[i] / m32[i] if m32[i] > 0 else 0
    y = max(m64[i], m32[i]) + max(s64[i], s32[i]) + 0.05
    ax.text(x[i], y, f"{sp:.1f}×", ha="center", fontsize=7, color="#666")

ax.set_ylabel("Time (ms)")
ax.set_xticks(x)
ax.set_xticklabels(op_names, fontsize=8, rotation=15, ha="right")
ax.legend()
ax.set_title(f"Single-point timing ({jax.default_backend()})")
fig.tight_layout()
plt.show()

## Numerical accuracy: float32 vs float64

In [ ]:
# Use the models already created above (separate cells, no mixing)
rng = np.random.default_rng(42)
n_test = 50
errors = {"W": [], "DW": [], "hessian": []}

for i in range(n_test):
    z_np = rng.uniform(-0.3, 0.3, 2) + 1j * rng.uniform(2, 5, 2)
    tau_np = complex(rng.uniform(-0.3, 0.3), rng.uniform(2, 5))
    fl_np = rng.integers(-5, 6, 12).astype(float)
    
    # float64 (model_64 was compiled with float64 inputs)
    jvc.set_precision("float64")
    z_a = jnp.array(z_np); tau_a = jnp.array(tau_np); fl_a = jnp.array(fl_np)
    W_ref = np.complex128(model_64.superpotential(z_a, tau_a, fl_a))
    DW_ref = np.array(model_64.DW(z_a, jnp.conj(z_a), tau_a, jnp.conj(tau_a), fl_a), dtype=np.complex128)
    H_ref = np.array(model_64.hessian(z_a, jnp.conj(z_a), tau_a, jnp.conj(tau_a), fl_a), dtype=np.complex128)
    
    # float32 (model_32 was compiled with float32 inputs)
    jvc.set_precision("float32")
    z_b = jnp.array(z_np); tau_b = jnp.array(tau_np); fl_b = jnp.array(fl_np)
    W_32 = np.complex128(model_32.superpotential(z_b, tau_b, fl_b))
    DW_32 = np.array(model_32.DW(z_b, jnp.conj(z_b), tau_b, jnp.conj(tau_b), fl_b), dtype=np.complex128)
    H_32 = np.array(model_32.hessian(z_b, jnp.conj(z_b), tau_b, jnp.conj(tau_b), fl_b), dtype=np.complex128)
    
    errors["W"].append(float(abs(W_32 - W_ref) / max(abs(W_ref), 1e-30)))
    errors["DW"].append(float(np.max(np.abs(DW_32 - DW_ref)) / max(np.max(np.abs(DW_ref)), 1e-30)))
    errors["hessian"].append(float(np.max(np.abs(H_32 - H_ref)) / max(np.max(np.abs(H_ref)), 1e-30)))

jvc.set_precision("float64")

print(f"Relative error (float32 vs float64) over {n_test} random points:")
print(f"{'Operation':>10s}   {'Mean':>12s}   {'Max':>12s}")
print("-" * 40)
for op, errs in errors.items():
    e = np.array(errs)
    print(f"{op:>10s}   {e.mean():.4e}   {e.max():.4e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
colors = {"W": "#2563eb", "DW": "#f97316", "hessian": "#7c3aed"}

for ax, (op, errs) in zip(axes, errors.items()):
    log_e = np.log10(np.array(errs) + 1e-20)
    ax.hist(log_e, bins=20, color=colors[op], alpha=0.6, edgecolor="white", linewidth=0.3)
    ax.axvline(x=-7, color="#dc2626", ls="--", lw=0.8, label="float32 limit")
    ax.set_xlabel(r"$\log_{10}$(rel. error)")
    ax.set_ylabel("Count")
    ax.set_title(op)
    ax.legend(fontsize=7)

fig.suptitle("float32 vs float64 accuracy", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## Batch (vmap) performance

In [ ]:
N = 1000
rng = np.random.default_rng(42)

# float64 batch
jvc.set_precision("float64")
z_b64 = jnp.array(rng.uniform(-0.3, 0.3, (N, 2)) + 1j * rng.uniform(2, 5, (N, 2)))
tau_b64 = jnp.array(rng.uniform(-0.3, 0.3, N) + 1j * rng.uniform(2, 5, N))
fl_b64 = jnp.array(rng.integers(-5, 6, (N, 12)).astype(float))

print(f"float64: vmap over {N} points")
t_W64 = benchmark(vmap(model_64.superpotential), z_b64, tau_b64, fl_b64, label="vmap(W)")
t_DW64 = benchmark(vmap(model_64.DW), z_b64, jnp.conj(z_b64), tau_b64, jnp.conj(tau_b64), fl_b64, label="vmap(DW)")

In [ ]:
# float32 batch
jvc.set_precision("float32")
z_b32 = jnp.array(rng.uniform(-0.3, 0.3, (N, 2)) + 1j * rng.uniform(2, 5, (N, 2)))
tau_b32 = jnp.array(rng.uniform(-0.3, 0.3, N) + 1j * rng.uniform(2, 5, N))
fl_b32 = jnp.array(rng.integers(-5, 6, (N, 12)).astype(float))

print(f"float32: vmap over {N} points")
t_W32 = benchmark(vmap(model_32.superpotential), z_b32, tau_b32, fl_b32, label="vmap(W)")
t_DW32 = benchmark(vmap(model_32.DW), z_b32, jnp.conj(z_b32), tau_b32, jnp.conj(tau_b32), fl_b32, label="vmap(DW)")

jvc.set_precision("float64")

print(f"\nSpeedup: vmap(W) {t_W64[0]/t_W32[0]:.2f}×, vmap(DW) {t_DW64[0]/t_DW32[0]:.2f}×")

## Summary

| Aspect | float64 | float32 |
|--------|:-------:|:-------:|
| Precision | ~$10^{-15}$ | ~$10^{-7}$ |
| Speed (CPU) | Baseline | Comparable |
| Speed (CUDA GPU) | Baseline | Up to 2–4× |
| Newton tolerance | $10^{-12}$ | $10^{-6}$ |
| Recommended for | Publications, final results | Exploration, large scans |

**Key takeaways:**
- On CPU, float32 gives modest speedup (modern CPUs have efficient float64)
- On CUDA, float32 can be significantly faster (tensor cores, memory bandwidth)
- Metal (Apple GPU) doesn't support complex numbers — falls back to CPU
- Use `jvc.set_precision("float32")` for fast exploration, float64 for final results